# 🏆 CAPSTONE — Banking CRM Intelligence System
**Estimated time:** 10–15 hours · **This is a portfolio project — invest in doing it well.**

---

## System Architecture
```
┌─────────────────────────────────────────────────────────────────┐
│                   Banking CRM Intelligence System                │
├────────────┬────────────────────┬───────────────────────────────┤
│  INGESTION │  INTELLIGENCE CORE │  AGENT ORCHESTRATOR           │
│  (M1+M2)   │  (M3+M5)           │  (M4+M7)                      │
│            │                    │                               │
│  CFPB →    │  ChromaDB (RAG)    │  OrchestratorNode             │
│  ComplaintR│  XGBoost (churn)   │  ├─ PolicyRAGNode  ─┐        │
│  ecord     │  hybrid_route()    │  ├─ ChurnAnalysis ──┤ (∥)    │
│  trim 2k   │                    │  ├─ RetentionOffer ─┘        │
│  tokens    │                    │  └─ AuditNode (last)          │
├────────────┴────────────────────┴───────────────────────────────┤
│  OUTPUT & GUARDRAILS (M2+M7)  │  EVAL & MONITORING (M6)         │
│  CRMIntelligenceReport         │  Langfuse traces                │
│  Pydantic validation           │  RAGAS (policy Q&A)             │
│  HITL: offer > 100 PLN         │  LLM-as-judge (offers)          │
│  Audit trail                   │  capstone_eval_report.md        │
└────────────────────────────────────────────────────────────────-┘
```

**Datasets:**
- CFPB Consumer Complaints (HuggingFace)
- Telco Customer Churn (Kaggle / synthetic fallback)
- Synthetic bank statements (ReportLab, generated in M7)

**LLM mix:** LM Studio (local, $0) for bulk · Cloud API for high-value tasks


In [ ]:
# ── Capstone imports & env check ──────────────────────────────────
import os, json, time, datetime, re, asyncio
from pathlib import Path
from typing import Optional, List, Any
from pydantic import BaseModel, Field, field_validator
from dataclasses import dataclass, field as dc_field
import pandas as pd
import numpy as np

from openai import OpenAI
from anthropic import Anthropic
from datasets import load_dataset
from llm_checker import check, hint, evaluate, progress, show_exercise

openai_client = OpenAI()
claude_client = Anthropic()
lm_client     = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

# Check M8 artifacts exist
ARTIFACTS = {
    "complaints_extracted": Path("data/capstone/complaints_extracted.jsonl"),
    "hybrid_routing_log":   Path("data/capstone/hybrid_routing_log.jsonl"),
    "golden_eval":          Path("data/capstone/golden_eval.jsonl"),
}
for name, path in ARTIFACTS.items():
    status = "✅" if path.exists() else "⚠️  MISSING — run Module 8 first"
    print(f"  {status}  {name}: {path}")

print("\n✅ Capstone imports ready")

---
## Step 1 — Ingestion Layer (M1 + M2)
**Goal:** Load 100 CFPB complaints, classify with few-shot CoT, extract `ComplaintRecord` via Instructor, enforce 2000-token context window.

**Auto-check verifies:**
- ✓ 100 complaints loaded and classified
- ✓ Pydantic pass rate ≥ 90% logged
- ✓ Context window never exceeds 2000 tokens


In [ ]:
show_exercise(
    "CAP-1", "Ingestion Layer",
    "Load 100 CFPB complaints. Classify into CRM categories. Extract ComplaintRecord Pydantic model. "
    "Trim context to ≤ 2000 tokens. Log pass rate.",
    hints=[
        "▸ load_dataset('cfpb/us-consumer-finance-complaints', split='train').select(range(100))",
        "▸ Use async_classify() pattern from 01b for batch processing",
        "▸ Reuse ComplaintRecord and extract_complaint_record() from Module 8",
    ],
    checks=[
        "100 complaints loaded and classified",
        "Pydantic pass rate ≥ 90% logged",
        "Context window never exceeds 2000 tokens"
    ],
    exercise_type="EXERCISE"
)

In [ ]:
# ── Shared Pydantic models (used across all steps) ────────────────
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

def trim_to_budget(text: str, budget: int = 1800) -> str:
    tokens = enc.encode(text)
    return enc.decode(tokens[:budget]) if len(tokens) > budget else text

class ComplaintRecord(BaseModel):
    customer_id:       str
    category:          str
    sentiment:         str
    urgency:           str
    product_mentioned: str
    action_required:   bool
    summary:           str = Field(max_length=200)

class ChurnRiskCard(BaseModel):
    customer_id:  str
    churn_proba:  float
    risk_tier:    str   # low / medium / high
    top_features: List[str]

class PolicySnippet(BaseModel):
    query:   str
    content: str
    source:  str = "ChromaDB"

class RetentionOffer(BaseModel):
    customer_id:      str
    offer_text:       str
    offer_value_pln:  float
    approved:         bool = False

class AuditEntry(BaseModel):
    ts:      str
    agent:   str
    action:  str
    summary: str
    blocked: bool = False

class CRMIntelligenceReport(BaseModel):
    customer_id:        str
    complaint:          ComplaintRecord
    churn_risk:         ChurnRiskCard
    policy_match:       PolicySnippet
    retention_offer:    Optional[RetentionOffer]
    audit_trail:        List[AuditEntry]
    processing_time_ms: int
    total_cost_usd:     float

print("✅ Pydantic schemas ready")

In [ ]:
# ── Step 1 implementation ──────────────────────────────────────────
ds = load_dataset("cfpb/us-consumer-finance-complaints", split="train").select(range(100))
complaints_df = pd.DataFrame(ds).fillna("")

INGESTED_RECORDS: List[ComplaintRecord] = []
ingestion_failures = []

# Load from Module 8 cache if available
if ARTIFACTS["complaints_extracted"].exists():
    with open(ARTIFACTS["complaints_extracted"]) as f:
        for line in f:
            d = json.loads(line)
            try:
                INGESTED_RECORDS.append(ComplaintRecord(**d))
            except Exception:
                pass
    print(f"✅ Loaded {len(INGESTED_RECORDS)} records from M8 cache")
else:
    print("⚠️  M8 cache not found — running ingestion from scratch (slower)")

    def extract_complaint(row: pd.Series) -> Optional[ComplaintRecord]:
        text_col = "complaint_what_happened" if "complaint_what_happened" in row.index else "consumer_complaint_narrative"
        text = trim_to_budget(row.get(text_col, ""))
        if not text.strip():
            return None
        prompt = (
            f"Classify this CFPB complaint. Reply ONLY valid JSON.\n\n"
            f"COMPLAINT: {text}\n\n"
            '{"category":"credit_card|mortgage|checking_account|loan|other",'
            '"sentiment":"positive|neutral|negative","urgency":"low|medium|high",'
            '"product_mentioned":"string","action_required":true|false,'
            '"summary":"one sentence ≤200 chars"}'
        )
        try:
            msg = lm_client.chat.completions.create(
                model="local-model", max_tokens=250,
                messages=[{"role":"user","content":prompt}]
            )
            raw = msg.choices[0].message.content.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            return ComplaintRecord(customer_id=str(row.name), **json.loads(raw))
        except Exception as e:
            ingestion_failures.append({"id": str(row.name), "error": str(e)})
            return None

    for _, row in complaints_df.iterrows():
        rec = extract_complaint(row)
        if rec:
            INGESTED_RECORDS.append(rec)

INGESTION_PASS_RATE = len(INGESTED_RECORDS) / 100
print(f"\n📊 Ingestion stats:")
print(f"  Records: {len(INGESTED_RECORDS)}/100")
print(f"  Pass rate: {INGESTION_PASS_RATE:.1%}")
print(f"  Failures: {len(ingestion_failures)}")
print(f"  Token violations: 0 (enforced by trim_to_budget)")

In [ ]:
# ── Auto-check Step 1 ─────────────────────────────────────────────
check([
    (len(INGESTED_RECORDS) >= 90,     "≥ 90 complaints extracted (of 100)"),
    (INGESTION_PASS_RATE >= 0.90,     f"Pass rate ≥ 90% (got {INGESTION_PASS_RATE:.1%})"),
    (all(isinstance(r, ComplaintRecord) for r in INGESTED_RECORDS), "All records are ComplaintRecord"),
    (all(r.customer_id != "" for r in INGESTED_RECORDS), "All records have customer_id"),
], exercise_id="CAP-1")

---
## ✅ Step 1 Complete
- ☐ 100 complaints loaded and classified
- ☐ Pydantic pass rate ≥ 90% logged
- ☐ Context window never exceeds 2000 tokens

➡️ Continue to `lesson_09b_intelligence_core.ipynb`